# Sprint 4 — Sesión Práctica Consolidada (70 min)
## Funnels, CTEs, A/B Testing e impacto de negocio con SQL en Google Colab

**Modalidad:** coding together + breakout rooms  
**Motor:** DuckDB en Google Colab  
**Datos:** archivos `.parquet` cargados desde GitHub, carpeta `/datasets`  
**Prefijo de archivos:** `sprint04_`

> Esta sesión consolida los contenidos prácticos del Sprint 4: construcción de funnels, segmentación, simulación de mejoras y comunicación de impacto de negocio.

<div style="text-align: center">
    <img src="https://raw.githubusercontent.com/ljpiere/tpdata_python/main/images/w1s1_2.png" width="400">
</div>

## Estructura y timeboxing

| Tiempo | Bloque | Modalidad | Contenido |
|---:|---|---|---|
| 0–8 | Bloque 0 | Setup guiado | Cargar tablas desde GitHub y verificar datos |
| 8–18 | Bloque 1 | Breakout Rooms | Exploración inicial y checks rápidos |
| 18–35 | Bloque 2 | Coding Together | Funnel base por variante A/B |
| 35–50 | Bloque 3 | Coding Together | Simulación de mejora en checkout |
| 50–60 | Bloque 4 | Breakout Rooms | Segmentos, tablero ejecutivo y recomendación |
| 60–70 | Cierre | Plenaria | Preguntas de validación, takeaways y canales de ayuda |

## Objetivos de aprendizaje

Al finalizar la práctica, el/la estudiante podrá:

1. Cargar tablas `.parquet` desde GitHub en Google Colab.
2. Calcular un funnel de conversión usando SQL y CTEs.
3. Comparar tasas de conversión entre variantes A/B.
4. Simular una mejora en una etapa del funnel y estimar impacto en revenue.
5. Construir una recomendación ejecutiva basada en evidencia.

---
# Bloque 0 · Preparación del entorno

Usaremos las mismas URLs raw de GitHub:

```text
https://github.com/ljpiere/tpdata_course/raw/refs/heads/main/new_DA/DA_tutor/datasets/sprint04_*.parquet
```

Archivos usados:

- `sprint04_users.parquet`
- `sprint04_events.parquet`
- `sprint04_funnel_sessions.parquet`
- `sprint04_experiments.parquet`

In [ ]:
# Instalación e importación de librerías
!pip install -q duckdb pandas pyarrow requests

import io
import requests
import pandas as pd
import duckdb

BASE_URL = "https://github.com/ljpiere/tpdata_course/raw/refs/heads/main/new_DA/DA_tutor/datasets"

DATASETS = {
    "users": "sprint04_users.parquet",
    "events": "sprint04_events.parquet",
    "funnel_sessions": "sprint04_funnel_sessions.parquet",
    "experiments": "sprint04_experiments.parquet",
}

def read_parquet_from_github(file_name: str) -> pd.DataFrame:
    url = f"{BASE_URL}/{file_name}"
    response = requests.get(url)
    response.raise_for_status()
    return pd.read_parquet(io.BytesIO(response.content))

# Cargar archivos parquet desde GitHub
users = read_parquet_from_github(DATASETS["users"])
events = read_parquet_from_github(DATASETS["events"])
funnel_sessions = read_parquet_from_github(DATASETS["funnel_sessions"])
experiments = read_parquet_from_github(DATASETS["experiments"])

# Crear conexión DuckDB en memoria
con = duckdb.connect(database=":memory:")

# Registrar DataFrames como tablas SQL
con.register("users_df", users)
con.register("events_df", events)
con.register("funnel_sessions_df", funnel_sessions)
con.register("experiments_df", experiments)

con.execute("CREATE OR REPLACE TABLE users AS SELECT * FROM users_df")
con.execute("CREATE OR REPLACE TABLE events AS SELECT * FROM events_df")
con.execute("CREATE OR REPLACE TABLE funnel_sessions AS SELECT * FROM funnel_sessions_df")
con.execute("CREATE OR REPLACE TABLE experiments AS SELECT * FROM experiments_df")

def sql(query: str) -> pd.DataFrame:
    """Ejecuta una consulta SQL en DuckDB y devuelve un DataFrame."""
    return con.execute(query).df()

print("Tablas creadas en DuckDB:", ", ".join(DATASETS.keys()))

## 0.1 Verificación de carga

In [ ]:
sql("""
SELECT 'users' AS table_name, COUNT(*) AS rows FROM users
UNION ALL
SELECT 'events' AS table_name, COUNT(*) AS rows FROM events
UNION ALL
SELECT 'funnel_sessions' AS table_name, COUNT(*) AS rows FROM funnel_sessions
UNION ALL
SELECT 'experiments' AS table_name, COUNT(*) AS rows FROM experiments;
""")

---
# Bloque 1 · Breakout Rooms — Exploración inicial (10 min)

Trabaja con una pareja o triada. El objetivo es entender la estructura antes de calcular métricas.

### Ejercicio 1.1 — Vista rápida de sesiones de funnel

In [ ]:
sql("""
SELECT *
FROM funnel_sessions
ORDER BY session_date, session_id
LIMIT 10;
""")

### Ejercicio 1.2 — Balance de sesiones por variante

Antes de comparar A vs. B, revisa si los grupos tienen tamaños razonables.

In [ ]:
sql("""
SELECT
    variant,
    COUNT(*) AS n_sessions,
    COUNT(DISTINCT user_id) AS n_users,
    SUM(purchased) AS purchases,
    SUM(revenue) AS revenue
FROM funnel_sessions
GROUP BY variant
ORDER BY variant;
""")

### Ejercicio 1.3 — Metadatos del experimento

In [ ]:
sql("""
SELECT *
FROM experiments
ORDER BY experiment_id, variant;
""")

---
# Bloque 2 · Coding Together — Funnel base por variante A/B (17 min)

Construiremos un resumen por variante con los conteos de cada etapa:

`viewed_product → added_to_cart → begin_checkout → purchased`

La tabla principal es `funnel_sessions`. Cada fila representa una sesión.

### Ejercicio 2.1 — Conteos del funnel por variante

In [ ]:
sql("""
WITH funnel AS (
    SELECT
        variant,
        COUNT(*) AS n_sessions,
        SUM(viewed_product) AS n_view,
        SUM(added_to_cart) AS n_cart,
        SUM(begin_checkout) AS n_checkout,
        SUM(purchased) AS n_purchase,
        SUM(revenue) AS total_revenue
    FROM funnel_sessions
    GROUP BY variant
)
SELECT *
FROM funnel
ORDER BY variant;
""")

### Ejercicio 2.2 — Tasas de conversión por etapa

Calcula tasas entre etapas para identificar dónde se pierde más tráfico.

In [ ]:
sql("""
WITH funnel AS (
    SELECT
        variant,
        COUNT(*) AS n_sessions,
        SUM(viewed_product) AS n_view,
        SUM(added_to_cart) AS n_cart,
        SUM(begin_checkout) AS n_checkout,
        SUM(purchased) AS n_purchase,
        SUM(revenue) AS total_revenue
    FROM funnel_sessions
    GROUP BY variant
)
SELECT
    variant,
    n_sessions,
    n_view,
    n_cart,
    n_checkout,
    n_purchase,
    ROUND(n_cart * 1.0 / NULLIF(n_view, 0), 3) AS view_to_cart_rate,
    ROUND(n_checkout * 1.0 / NULLIF(n_cart, 0), 3) AS cart_to_checkout_rate,
    ROUND(n_purchase * 1.0 / NULLIF(n_checkout, 0), 3) AS checkout_to_purchase_rate,
    ROUND(n_purchase * 1.0 / NULLIF(n_view, 0), 3) AS end_to_end_rate,
    ROUND(total_revenue, 2) AS total_revenue
FROM funnel
ORDER BY variant;
""")

### Discusión rápida

Preguntas para plenaria:

- ¿Qué variante convierte mejor al final del funnel?
- ¿En qué etapa parece estar el mayor cuello de botella?
- ¿Qué métrica usarías para recomendar una mejora: conversión final, revenue o ambas?

---
# Bloque 3 · Coding Together — Simular mejora en checkout (15 min)

El equipo de producto propone mejorar la variante B para aumentar la conversión de `begin_checkout` a `purchased` en **15% relativo**.

Ejemplo: si la tasa actual es 0.50, la tasa simulada sería 0.575.

Regla: la cantidad de compras simuladas no puede superar la cantidad de sesiones que llegaron a checkout.

### Ejercicio 3.1 — Escenario actual vs. simulado

In [ ]:
sql("""
WITH funnel AS (
    SELECT
        variant,
        SUM(begin_checkout) AS n_checkout,
        SUM(purchased) AS n_purchase,
        SUM(revenue) AS actual_revenue,
        AVG(CASE WHEN purchased = 1 THEN revenue END) AS avg_order_value
    FROM funnel_sessions
    GROUP BY variant
),
simulacion AS (
    SELECT
        variant,
        n_checkout,
        n_purchase AS actual_purchases,
        actual_revenue,
        avg_order_value,
        CASE
            WHEN variant = 'B'
                THEN LEAST(n_checkout, ROUND(n_purchase * 1.15))
            ELSE n_purchase
        END AS simulated_purchases
    FROM funnel
)
SELECT
    variant,
    n_checkout,
    actual_purchases,
    simulated_purchases,
    simulated_purchases - actual_purchases AS incremental_purchases,
    ROUND(actual_revenue, 2) AS actual_revenue,
    ROUND(simulated_purchases * avg_order_value, 2) AS simulated_revenue,
    ROUND((simulated_purchases * avg_order_value) - actual_revenue, 2) AS incremental_revenue
FROM simulacion
ORDER BY variant;
""")

### Ejercicio 3.2 — Interpretación del escenario

Convierte la tabla anterior en una lectura de negocio:

- **Condición:** qué está pasando hoy.
- **Findings:** qué cambia en el escenario simulado.
- **Insight:** qué acción recomendarías y qué métrica monitorearías.

---
# Bloque 4 · Breakout Rooms — Segmentos, tablero ejecutivo y recomendación (10 min)

Ahora cruzamos `funnel_sessions` con `users` para ver si el impacto cambia por segmento.

### Ejercicio 4.1 — Funnel por segmento y variante

In [ ]:
sql("""
WITH base AS (
    SELECT
        u.segment,
        u.country,
        fs.variant,
        fs.viewed_product,
        fs.added_to_cart,
        fs.begin_checkout,
        fs.purchased,
        fs.revenue
    FROM funnel_sessions fs
    JOIN users u
        ON fs.user_id = u.user_id
),
agg AS (
    SELECT
        segment,
        variant,
        COUNT(*) AS n_sessions,
        SUM(viewed_product) AS n_view,
        SUM(added_to_cart) AS n_cart,
        SUM(begin_checkout) AS n_checkout,
        SUM(purchased) AS n_purchase,
        SUM(revenue) AS revenue
    FROM base
    GROUP BY segment, variant
)
SELECT
    segment,
    variant,
    n_sessions,
    n_view,
    n_cart,
    n_checkout,
    n_purchase,
    ROUND(n_purchase * 1.0 / NULLIF(n_view, 0), 3) AS end_to_end_rate,
    ROUND(revenue, 2) AS revenue
FROM agg
ORDER BY segment, variant;
""")

### Ejercicio 4.2 — Recomendación ejecutiva

Con base en la tabla anterior, redacta una recomendación breve:

```text
Recomendamos priorizar __________ porque __________.
La métrica principal a monitorear sería __________.
El riesgo o validación adicional necesaria es __________.
```

Este ejercicio entrena la traducción de resultados SQL a decisiones de negocio.

---
# Cierre · Validación de conocimiento y takeaways (10 min)

## Preguntas de validación para el estudiante

1. ¿Por qué se recomienda validar el balance de sesiones por variante antes de interpretar un A/B test?
2. ¿Cuál es la diferencia entre `n_purchase` y `checkout_to_purchase_rate`?
3. ¿Por qué usamos `NULLIF` al calcular tasas?
4. ¿Qué significa que una mejora sea de 15% relativo y no de 15 puntos porcentuales?
5. ¿Por qué segmentar por `segment` puede cambiar la recomendación final?

## Respuestas esperadas / criterios

1. Porque grupos muy desbalanceados pueden hacer que la comparación sea menos confiable.
2. `n_purchase` es conteo absoluto; `checkout_to_purchase_rate` es proporción sobre quienes llegaron a checkout.
3. Para evitar división por cero cuando una etapa no tiene usuarios.
4. 15% relativo multiplica la tasa actual por 1.15; 15 puntos porcentuales suma 0.15 directamente.
5. Porque el efecto puede ser fuerte en un segmento y débil o negativo en otro.

## Takeaways de la sesión

- Las CTEs permiten construir funnels de forma ordenada y auditable.
- Un A/B test debe interpretarse con conteos, tasas y contexto de negocio.
- La simulación ayuda a estimar impacto, pero no reemplaza una validación experimental completa.
- Revenue y conversión deben analizarse juntos.
- La segmentación evita conclusiones demasiado generales.

### ¿Necesitas ayuda?

Recuerda los canales de ayuda disponibles:

- `DATA-CO-LEARNING`: revisa los horarios de atención en la guía del estudiante. Hay horarios de apoyo para dudas puntuales.
- `DA_CONSULTA`: publica preguntas sobre SQL, errores de Colab, carga de archivos o interpretación de resultados.
- Para pedir ayuda, comparte siempre: consulta SQL, error completo, tabla usada y captura del resultado esperado.

## Siguientes pasos

Después de esta sesión, el/la estudiante debería practicar variaciones del mismo análisis:

- cambiar la etapa simulada,
- analizar por país,
- comparar canal de adquisición,
- exportar resultados a Google Sheets para construir una visualización ejecutiva.